# 5.2 Reproducibility via Code Notebooks _and_
# 5.4 Best Practices for Reproducible Coding 

5.2 Create a notebook that does some simple EDA on some data. It should generate at **least one plot and at least one table**. Create a script to download the CDC dataset. Upload your work

5.4 Add documentation to your simple EDA notebook. Create a Makefile similar to the one shown in class (e.g., with commands to download the data, run the analysis, etc.). Upload your analysis file.

# Setup

This setup uses code from Dr. Rampure to import libraries and define functions for working with the CDC data. 

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio

from IPython.display import display
from sklearn.compose import ColumnTransformer, make_column_transformer, make_column_selector
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import KFold, GridSearchCV, cross_validate, train_test_split
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, PolynomialFeatures, StandardScaler

ROOT = Path("/Users/higgins/Documents/UWGrad/Conferences_Apps_Projects/DAIR3/DAIR3-Workshop/resources/unit_4")
DATA_DIR = ROOT / "data"
IMAGE_DIR = ROOT / "images"
RANDOM_STATE = 42

pd.options.plotting.backend = "plotly"
pio.templates["palatino_white"] = go.layout.Template(pio.templates["plotly_white"])
pio.templates["palatino_white"].layout.font.family = "Palatino, Palatino Linotype, Book Antiqua, serif"
pio.templates["palatino_white"].data.scatter = [go.Scatter(marker={"size": 8})]
pio.templates["palatino_white"].data.scattergl = [go.Scattergl(marker={"size": 8})]
pio.templates.default = "palatino_white"
px.defaults.template = "palatino_white"
px.defaults.width = 840
px.defaults.height = 460


def display_df(df, rows=10):
    return display(df.head(rows))


def make_polynomial_sample(n=80, random_state=23):
    rng = np.random.default_rng(random_state)
    x = np.linspace(-5, 5, n)
    y = 2 + 0.8 * x - 0.35 * x ** 2 + 0.08 * x ** 3 + rng.normal(0, 3, n)
    return pd.DataFrame({"x": x, "y": y})


def one_hot_encoder(**kwargs):
    try:
        return OneHotEncoder(sparse_output=False, **kwargs)
    except TypeError:
        return OneHotEncoder(sparse=False, **kwargs)


def find_birthweight_file(year=1971):
    candidates = [
        DATA_DIR / f"{year}.csv.gz",
        ROOT.parent / "DAIR3-Workshop" / "resources" / "unit_3" / "data" / f"{year}.csv.gz",
        ROOT.parent / "resources" / "unit_3" / "data" / f"{year}.csv.gz",
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate
    searched = "\n".join(f"- {candidate}" for candidate in candidates)
    raise FileNotFoundError(
        "Could not find the prepared NCHS birthweight CSV. "
        "Run the Unit 3 prep.py script first, or place 1971.csv.gz in materials-draft/data.\n"
        f"Searched:\n{searched}"
    )


def load_birthweight_1971():
    return pd.read_csv(find_birthweight_file(1971))


def prepare_birthweight_modeling_data():
    births = load_birthweight_1971()
    cols = ["birthweight", "sex", "momage", "dadage", "plurality", "birthorder"]
    return births[cols].dropna().copy()


In [ ]:
try:
    births = load_birthweight_1971()
    print(f"Loaded {births.shape[0]:,} births and {births.shape[1]:,} columns.")
    display(births.sample(5))
except FileNotFoundError as err:
    births = None
    print(err)


# Descriptives 

Look at a descriptive table, examining the outcome grouped by plurality (assuming this is of interest to subsequent analysis questions) 

In [ ]:
print(births.groupby("plurality")['birthweight'].describe())


Plot birthweight distribution by plurality 

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 6))

sns.violinplot(
    data=births,
    x="plurality",
    y="birthweight"
)

plt.xticks(rotation=45, ha="right")
plt.xlabel("Plurality")
plt.ylabel("Mean birthweight, grams")
plt.title("Mean Birthweight by Plurality with 95% Confidence Intervals")
plt.tight_layout()
plt.show()